In [40]:
from sentence_transformers import SentenceTransformer

from src.embeddings.embedder import Embedder

# 1. Load a pretrained Sentence Transformer model
sentence_transformer = SentenceTransformer("all-MiniLM-L6-v2")


In [32]:
embedder = Embedder()

In [12]:
embedder.initiate_qdrant_collection()

In [19]:
from pathlib import Path

path_pdf = "../cv_pdfs/ACCOUNTANT/10554236.pdf"

pdf_path = Path(path_pdf)
cv_id = pdf_path.name
print(cv_id)

10554236.pdf


In [15]:
text = embedder.extract_text_from_pdf(path_pdf)


In [18]:
chunks_list = embedder.simple_chunking(text)


In [26]:
import uuid
from qdrant_client.http.models import PointStruct

payloads_list = []
for chunk_index, chunk in enumerate(chunks_list):
    # Payload = métadonnées
    payload = {
        "cv_id": cv_id,
        "chunk_index": chunk_index,
        "text": chunk,
    }
    payloads_list.append(payload)

embedded_chunks = embedder.sentence_transformer.encode(chunks_list, convert_to_numpy=True)

points = []
for i, (vec, payload) in enumerate(zip(embedded_chunks, payloads_list)):
    point = PointStruct(
        id=str(uuid.uuid4()),
        vector=vec.tolist(),  # Qdrant attend une liste Python
        payload=payload,
    )
    points.append(point)

In [23]:
print(payloads_list[1])

{'cv_id': '10554236.pdf', 'chunk_index': 1, 'text': 'countant\nCity , State\nEnterprise Resource Planning Office (ERO)\nIn this position as an Accountant assigned to the Defense Enterprise Accounting and Management System (DEAMS) ERO I was\nresponsible for identifying and resolving issues affecting the DEAMS General Ledger.\nI worked with teammates from the Procure to Pay, Orders to Cash, and Budget to Report areas to resolve daily challenges encountered\nwith the deployment of DEAMS to additional customers and when system change requests were promoted to production.\nI supported the testing of scripts, patches, and system change requests ensuring any anomalies were identified to the DEAMS Functional\nManagement Office for action by the DEAMS Program Management Office and/or the System Integrator.\nIn addition, I served on a tiger team designed to identify and resolve General Ledger posting differences and supported the development of\n$360B in accounting adjustments allowing for the f

In [24]:
print(points[1])

id='10554236.pdf_1' vector=[-0.03456282243132591, 0.03306715935468674, 0.03223037347197533, -0.017789997160434723, -0.08047565072774887, -0.04732902720570564, 0.03492509573698044, -0.0010272110812366009, -0.03835217282176018, 0.052386462688446045, 0.01286333054304123, -0.031246831640601158, -0.05280787870287895, -0.013279403559863567, 0.03844079747796059, 0.02401624247431755, 0.05746904015541077, -0.01582379639148712, 0.018744036555290222, 0.039730679243803024, -0.024633515626192093, -0.04756610095500946, -0.10901130735874176, 0.06553022563457489, -0.05914667621254921, -0.016940709203481674, 0.013907600194215775, 0.04250716045498848, -0.09633800387382507, -0.1467202752828598, -0.02517249621450901, -0.027651341632008553, 0.07192160934209824, 0.008812038227915764, 0.09810580313205719, 0.010421138256788254, 0.009645331650972366, 0.011198142543435097, 0.012525023892521858, -0.09985069930553436, 0.01790234073996544, -0.018174128606915474, -0.016656208783388138, -0.040202800184488297, -0.010

In [27]:
embedder.qdrant_client.upsert(
        collection_name="cv_collection",
        points=points,
    )

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [34]:
info = embedder.qdrant_client.get_collection("cv_collection")
print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=31 segments_count=4 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None,

In [33]:
embedder.index_single_cv("../cv_pdfs/ACCOUNTANT/10674770.pdf")

📄 Indexation du CV : ..\cv_pdfs\ACCOUNTANT\10674770.pdf (cv_id=10674770.pdf)


In [41]:
embedder_2 = Embedder()

In [42]:
embedder_2.index_single_cv("../cv_pdfs/ACCOUNTANT/10674770.pdf")

📄 Indexation du CV : ..\cv_pdfs\ACCOUNTANT\10674770.pdf (cv_id=10674770.pdf)
